# Just in case we need to modulate time format of the collection time


In [ ]:

import pandas as pd
import re
from pathlib import Path

# here is where fernando can adjust the file path and the hours need to shift
IN_PATH = Path(r"../src/test_data/AlgorithmData-Acceleration-[20251001-20251031]-part2.csv")   
OUT_PATH = IN_PATH.with_name(IN_PATH.stem + "_SHIFTED_Z.csv")

TIME_COL = "Collecting time"  
SHIFT_HOURS = 1                # +1 forward, -2 backward, etc.


ALLOW_BAD_ROWS = True


_iso_z_re_ms_or_more = re.compile(r"^(?P<prefix>.*\.(?P<ms>\d{3}))(?P<extra>\d*)Z$")
_iso_has_fraction = re.compile(r"^(.*\.)\d+(Z)$")

def normalise_to_iso_z(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "nan":
        return None

    # Standardise separator
    x = x.replace(" ", "T")

    # Convert UTC offsets to Z (handles +00:00 and +0000)
    if x.endswith("+00:00"):
        x = x[:-6] + "Z"
    elif x.endswith("+0000"):
        x = x[:-5] + "Z"
    elif x.endswith("Z") is False:
        # If it has some other timezone, we still try parsing directly later,
        # but most of the data is UTC seemingly
        pass

    # Ensure a fractional part exists (milliseconds). If none, add .000
    if "T" in x and ("." not in x) and x.endswith("Z"):
        x = x[:-1] + ".000Z"

    # If fraction exists but is shorter than 3, pad to 3
    if x.endswith("Z") and "." in x:
        # Split just before Z
        base = x[:-1]
        left, frac = base.split(".", 1)
        # keep only digits in frac
        frac_digits = re.sub(r"\D", "", frac)
        if len(frac_digits) >= 3:
            frac3 = frac_digits[:3]
        else:
            frac3 = frac_digits.ljust(3, "0")
        x = f"{left}.{frac3}Z"

    # If more than 3 digits exist before Z (microseconds), trim to 3
    m = _iso_z_re_ms_or_more.match(x)
    if m:
        x = m.group("prefix") + "Z"

    return x

def format_iso_z_from_utc(dt_utc: pd.Series) -> pd.Series:
    # dt_utc is tz-aware UTC
    # %f = microseconds (6 digits). Slice to milliseconds.
    return dt_utc.dt.strftime("%Y-%m-%dT%H:%M:%S.%f").str.slice(0, 23) + "Z"


df = pd.read_csv(IN_PATH)
if TIME_COL not in df.columns:
    raise KeyError(f"Column '{TIME_COL}' not found. Columns: {df.columns.tolist()}")

raw = df[TIME_COL].astype("string")


norm = raw.map(normalise_to_iso_z)

# Parse as UTC. If strings already have Z -> ok. If some have +00:00 we normalised to Z.
dt = pd.to_datetime(norm, errors="coerce", utc=True)

bad_mask = dt.isna() & raw.notna() & (raw.str.strip() != "") & (raw.str.lower() != "nan")
bad_count = int(bad_mask.sum())

if bad_count and not ALLOW_BAD_ROWS:
    cols_to_show = [c for c in ["UUID", "Transmitting time", TIME_COL] if c in df.columns]
    display(df.loc[bad_mask, cols_to_show].head(50))
    raise ValueError(
        f"{bad_count} rows in '{TIME_COL}' could not be parsed. "
        "Set ALLOW_BAD_ROWS=True to keep them blank, or inspect the displayed rows."
    )


dt_shift = dt + pd.to_timedelta(SHIFT_HOURS, unit="h")


shifted_str = format_iso_z_from_utc(dt_shift)

# Keep blanks where parsing failed / original was blank
shifted_str = shifted_str.where(~dt_shift.isna(), other="")

# Write back
df[TIME_COL] = shifted_str


print("Input:", IN_PATH)
print("Output:", OUT_PATH)
print("Shift hours:", SHIFT_HOURS)
print("Total rows:", len(df))
print("Unparseable timestamps (left blank):", bad_count)

print("\nPreview (first 10):")
display(df[[TIME_COL]].head(10))

df.to_csv(OUT_PATH, index=False)
print("\nSaved")
